## Imports

In [18]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, MinMaxScaler, PolynomialFeatures, RobustScaler, OrdinalEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score, GridSearchCV, train_test_split
from sklearn.datasets import fetch_california_housing, load_iris
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, mean_squared_error, r2_score, mean_absolute_error
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.impute import SimpleImputer
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import random
import seaborn as sns



In [ ]:
# Create pipeline
pipeline = Pipeline([
    ('scaler', StandardScaler()),           # Step 1: Normalize features
    ('classifier', RandomForestClassifier()) # Step 2: Train model
])

# Train and evaluate with cross-validation
scores = cross_val_score(pipeline, X, y, cv=5) # $ for training, 1 for testing.
print(f"Average accuracy: {scores.mean():.3f}")

In [14]:
cal = fetch_california_housing()
X, y = cal.data, cal.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [ ]:
model = LinearRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print("MSE:", mean_squared_error(y_test, y_pred)) 
print("R2:", r2_score(y_test, y_pred))

## Simple Pipeline Implementation

In [ ]:
iris = load_iris()
X, y = iris.data, iris.target


pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', RandomForestClassifier())
])

# HYPERPARAMETER GRID (tune these 3 params)
param_grid = {
    'classifier__n_estimators': [50, 100, 200],        # Trees: 50/100/200
    'classifier__max_depth': [None, 10, 20],           # Depth: unlimited/10/20
    'classifier__min_samples_split': [2, 5]            # Split: 2/5 samples
}

# GridSearch: Tests 3×3×2 = 18 combinations!
grid_search = GridSearchCV(
    pipeline, param_grid, cv=5, 
    scoring='accuracy', 
    n_jobs=-1  # Use all CPU cores!
)

# ONE LINE trains 18 models × 5 folds = 90 total trainings!
grid_search.fit(X, y)

# RESULTS
print(f"Best accuracy: {grid_search.best_score_:.3f}")
print(f"Best params: {grid_search.best_params_}")
print(f"All scores: {grid_search.cv_results_['mean_test_score']}")

# Linear Regression

In [ ]:
def evaluate_model(y_true, y_pred):
    print("MAE Score:", round(mean_absolute_error(y_test, y_pred), 4))
    print("MSE:", round(mean_squared_error(y_test, y_pred), 4))
    print("RMSE:", round(np.sqrt(mean_squared_error(y_test, y_pred)), 4))
    print("R2 Score:", round(r2_score(y_test, y_pred), 4))
    
evaluate_model(y_test, y_pred)

In [ ]:
fch = fetch_california_housing()
X, y = fch.data, fch.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
regr = LinearRegression()
regr.fit(X_train, y_train)
y_pred = regr.predict(X_test)

In [ ]:
plt.scatter(X[:, 0], y, alpha=0.5, color="blue")
plt.xlabel("Median Income")
plt.ylabel("Median House Value")
plt.title("Carlifornia housing")
plt.show()

In [ ]:
regr.singular_

## Polynomial Features

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

# Generate synthetic data
np.random.seed(0)
x = np.linspace(0, 10, 100)
y = np.sin(x) + np.random.normal(scale=0.2, size=x.shape)

# Split data into training and testing sets
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.3, random_state=0)

# Fit polynomial regression models of different degrees
degrees = [1, 3, 10, 15, 18, 20]
train_errors = []
test_errors = []

for degree in degrees:
    poly_features = PolynomialFeatures(degree=degree)
    x_poly_train = poly_features.fit_transform(x_train[:, np.newaxis])
    x_poly_test = poly_features.transform(x_test[:, np.newaxis])
    
    model = LinearRegression()
    model.fit(x_poly_train, y_train)
    
    train_predictions = model.predict(x_poly_train)
    test_predictions = model.predict(x_poly_test)
    
    train_errors.append(mean_squared_error(y_train, train_predictions))
    test_errors.append(mean_squared_error(y_test, test_predictions))

# Plot learning curves
plt.figure(figsize=(10, 6))
plt.plot(degrees, train_errors, label='Train Error', marker='o')
plt.plot(degrees, test_errors, label='Test Error', marker='o')
plt.title('Learning Curves')
plt.xlabel('Polynomial Degree')
plt.ylabel('Mean Squared Error')
plt.xticks(degrees)
plt.legend()
plt.grid(True)
plt.show()

# Comparison of Ridge, Lasso and ElasticNet. 

In [ ]:
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import make_regression
from sklearn.linear_model import Ridge, RidgeCV, Lasso, LassoCV, ElasticNet, ElasticNetCV
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error, root_mean_squared_error


ridge = Ridge(alpha=0.1)
ridgecv = RidgeCV(np.array([0.1, 0.2, 0.3, 0.4, 0.5, 0.6]))
lasso = Lasso(alpha=0.1)
lassocv = LassoCV(alphas=np.array([0.1, 0.2, 0.3, 0.4, 0.5, 0.6]))
eln = ElasticNet(alpha=0.1)
elncv = ElasticNetCV(alphas=np.array([0.1, 0.2, 0.3, 0.4, 0.5, 0.6]))

X, y, coeffs = make_regression(
    n_samples=200,
    n_features=20,
    n_informative=5,
    noise=15,
    coef=True,
    random_state=42
)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2)

ridge.fit(X_train, y_train)
ridgecv.fit(X_train, y_train)
lasso.fit(X_train, y_train)
lassocv.fit(X_train, y_train)
eln.fit(X_train, y_train)
elncv.fit(X_train, y_train)

r_pred = ridge.predict(X_test)
rcv_pred = ridgecv.predict(X_test)
l_pred = lasso.predict(X_test)
lcv_pred = lassocv.predict(X_test)
e_pred = eln.predict(X_test)
ecv_pred = elncv.predict(X_test)

print(f"Ridge R2:   {ridge.score(X_test, y_test):.6f}   Metrics MSE: {root_mean_squared_error(y_test, r_pred):.6f}")
print(f"RidgeCV R2: {ridgecv.score(X_test, y_test):.6f}   Metrics MSE: {root_mean_squared_error(y_test, rcv_pred):.6f} Best Alpha: {ridgecv.alpha_}")
print(f"Lasso R2:   {lasso.score(X_test, y_test):.6f}   Metrics MSE: {root_mean_squared_error(y_test, l_pred):.6f}")
print(f"LassoCV R2: {lassocv.score(X_test, y_test):.6f}   Metrics MSE: {root_mean_squared_error(y_test, lcv_pred):.6f} Best Alpha: {lassocv.alpha_}")
print(f"Elnet R2:   {eln.score(X_test, y_test):.6f}   Metrics MSE: {root_mean_squared_error(y_test, e_pred):.6f}")
print(f"ElNetCV R2: {elncv.score(X_test, y_test):.6f}   Metrics MSE: {root_mean_squared_error(y_test, ecv_pred):.6f} Best Alpha: {elncv.alpha_}")

## Exercise
**Exercise 1** — Observe the sparsity effect. Using `make_regression(n_features=50, n_informative=8)`, fit `LassoCV` and print which feature indices survived (coef ≠ 0). Compare to the true informative indices.

**Exercise 2** — Alpha sensitivity. Loop over `alpha = [0.001, 0.01, 0.1, 1, 10, 100]` for `Ridge` and `Lasso`. Plot test R² vs alpha for both on the same chart. Notice how `Lasso` degrades faster at high alpha (it zeros too many features) vs `Ridge` (gentler decay).

**Exercise 3** — Correlated features. Build a dataset where features 0 and 1 are nearly identical `(X[:, 1] = X[:, 0] + small_noise)`. Fit `Lasso` vs `ElasticNet`. Observe how Lasso picks one and drops the other, while `ElasticNet` keeps both with similar small weights.

**Exercise 4** — `Pipeline`. Wrap your best model in an `sklearn.pipeline.Pipeline` with `StandardScaler` as the first step. This prevents data leakage during cross-validation (the scaler should be fit only on training folds, not the whole dataset).

``` python
from sklearn.pipeline import Pipeline
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('model',  ElasticNetCV(l1_ratio=[0.5, 0.9], cv=5, max_iter=10000))
])
pipe.fit(X_train_raw, y_train)   # pipe handles scaling internally
```


# Pandas Block, Exploratory Data Analysis(EDA)

In [ ]:

pd.DataFrame(
    {
        'head1': [1, 2, 3, 4, 5, 6, 7, 8, 4],
        'head2': [4, 9, 10, 11, 12, 13, 19, 20, 21]
    },
    dtype='int'
)

imp = SimpleImputer(strategy="most_frequent")
model = RandomForestClassifier(random_state=42)
df = sns.load_dataset('titanic')
# print(titanic.head())
adults = pd.read_csv('adult.csv')
athlete = pd.read_csv('athlete_events.csv')

In [55]:
X = df[['pclass', 'age', 'sibsp', 'parch']]
y = df["survived"]
X_train_imp = imp.fit_transform(X)

In [59]:
X_train, X_test, y_train, y_test = train_test_split(X_train_imp, y, test_size=0.2, random_state=42)
model.fit(X_train, y_train)
model.score(X_test, y_test)

# X_train_imp = imp.fit_transform(X_train)
# X_test_imp = imp.transform(X_test)
# ...

0.7206703910614525

In [60]:
model.fit(X_train, y_train)
print(model.score(X_test, y_test))

0.7206703910614525


In [ ]:

print(df.value_counts(), "\n")
# print(df.age.value_counts(), "\n")
# print(df.isnull().sum(), "\n") 
# print(df.describe(), "\n") # .info, .head(5), .tail(5), .columns, .index, .shape, .size, pd.to_csv, pd.to_parquet etc, .sample(10)
# print(df.shape, "\n")
cor = df.corr(numeric_only=True)
sns.heatmap(
    cor,
    annot=True,
    
)

In [ ]:
df.index = df['sex']
df.drop('sex', axis=1, inplace=True)

In [7]:
df.loc[:]
ds = ['anagrams',
 'anscombe',
 'attention',
 'brain_networks',
 'car_crashes',
 'diamonds',
 'dots',
 'dowjones',
 'exercise',
 'flights',
 'fmri',
 'geyser',
 'glue',
 'healthexp',
 'iris',
 'mpg',
 'penguins',
 'planets',
 'seaice',
 'taxis',
 'tips',
 'titanic']

In [ ]:
df = sns.load_dataset('planets')
# df.fillna(0)
df.isnull().sum()
df.dropna()

In [5]:
cal = fetch_california_housing()
X, y = cal.data, cal.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [42]:
pipe = Pipeline([
    ('scale', StandardScaler()),
    ('regr', LinearRegression())
])

pipe.fit(X_train, y_train)
pipe.score(X_test, y_test)

0.9468960016420045

In [35]:
ex = MinMaxScaler().fit_transform(X_train)
round(np.nanmean(ex), 13) # mean of 0, std of 1
np.nanstd(ex)

np.float64(0.250592511783928)

In [3]:
iris = load_iris()
X, y = iris.data, iris.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

knc = KNeighborsClassifier(n_neighbors=5)
knc.fit(X_train, y_train)
# knc.score(X_test, y_test)
accuracy_score(y_test, knc.predict(X_test))

0.9666666666666667

In [73]:
pipe2 = Pipeline([
    ('scale', MinMaxScaler()),
    ('knn', KNeighborsClassifier(n_neighbors=5))
])

pipe2.fit(X_train, y_train)
print(f"{(pipe2.score(X_test, y_test) * 100):.0f}%")

93%


In [250]:
rng = np.random.default_rng(20)
X = np.linspace(21, 50, 15)
y = X*(-3) + np.round(rng.random(15)*20)
X[0], X[-1] = 10, 60
y[0], y[-1] = 0, -300
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
# plt.Figure(figsize=(16, 100))
# plt.xlabel("Numbers from one to 20")
# plt.ylabel("Formula")
# plt.title("My Custom data")
# plt.scatter(X, y)
# plt.show()

In [251]:
rbs = RobustScaler()
X_train_scaled = rbs.fit_transform(X_train.reshape(-1, 1))
X_test_scaled = rbs.transform(X_test.reshape(-1, 1))
regr2 = LinearRegression()
regr2.fit(X_train_scaled, y_train)
regr2.score(X_test_scaled.reshape(-1, 1), y_test)

0.66863266303817

In [ ]:
df = sns.load_dataset('titanic')
enc = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
mtr = enc.fit_transform(df[['sex']])
sex_col_names = enc.get_feature_names_out(['sex'])
newdf = pd.DataFrame(mtr, columns=sex_col_names)
final = pd.concat([df, newdf], axis=1)
final.drop('sex', axis=1, inplace=True)


In [2]:
import numpy as np
ar = np.array([1, 2, 3, 4, 5, 6, 6, 7, 8, 9, 10, 12])
ar[ar > np.mean(ar)]

array([ 7,  8,  9, 10, 12])

In [32]:
# df = pd.DataFrame({
#     'age': [34, None, 45, 29, 52],
#     'salary': [72000, 55000, None, 48000, 91000],
#     'city': ['London', 'Paris', 'London', 'Berlin', None],
#     'job_type': ['Engineer', 'Designer', None, 'Engineer', 'Manager'],
#     'bought': [1, 0, 1, 0, 1]
# })
df = sns.load_dataset('titanic')

In [38]:
numeric_pipe = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('scaler', StandardScaler())
])

categorical_pipe = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

numerical_cols = ['age', 'fare']
categorical_cols = ['sex', 'embarked', 'class']

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_pipe, numerical_cols),
    ('cat', categorical_pipe, categorical_cols)
],
    remainder='drop'
)

full_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LogisticRegression())
])

X = df[['age', 'fare', 'sex', 'embarked', 'class']]
y = df['survived']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

full_pipeline.fit(X_train, y_train)
pred = full_pipeline.predict(X_test)
scores = cross_val_score(full_pipeline, X, y, cv=5)


In [41]:

accuracy_score(y_test, pred)

0.7910447761194029

# Everything I've learnt so far. 
1. What Linear Regression does under the hood. 
- Formula, line of best fit, parameters, attributes etc. 
2. Regression Metrics - MSE, RMSE, MAE, R2
- What the metrics do under the hood too. 
3. Train/test split, overfitting and underfitting. 
4. Polynomial Regression
- PolynomialFeatures()
5. Regularization - Ridge, Lasso, ElasticNet and their CVs.
- Penalizes large coefficients. 
- The alpha hyperparameter. 
6. EDA 
- Pandas simple commands and checking information about a dataset. 
7. Imputing
- SimpleImputer
- fillna and dropna
8. Scaling
- StandardScaler, MinMaxScaler, RobustScaler
9. Encoding
- OrdinalEncoder, OneHotEncoder
10. ColumnTransformer
- Full Pipelines, preprocessing. 
11. Feature Engineering. 
- Log Transform, Combining columns et